# Youth Depression Risk Profiling Pipeline

This notebook combines the original preprocessing, Logistic Regression, Random Forest, and K-means clustering notebooks into a reusable pipeline.

## Purpose

This version is organized to match the Open Source SW Contribution requirement:

- Use a single top-level function rather than repeating code
- Compare preprocessing methods, models, model parameters, and evaluation metrics
- Find the top 5 combinations and the best combination
- Provide a user manual/specification in a style similar to Pandas and Scikit-learn


## User Manual

### Main Function

```python
run_depression_risk_pipeline(filepath, encoding="cp949", scoring="f1_score")
```

### Parameters

- `filepath`: path to the 2024 Youth Life Survey CSV file
- `encoding`: CSV encoding, default is `"cp949"`
- `scoring`: metric used to rank model combinations. Recommended value is `"recall"` because the project focuses on detecting respondents with depressive symptoms.

### Returns

The function returns a dictionary containing:

- `model_results`: full model comparison table
- `top5_combinations`: top five preprocessing/model/parameter combinations
- `best_combination`: best-performing combination
- `best_model`: fitted best model
- `feature_importance`: top feature importance table
- `clustering_result`: K-means clustering result and cluster-level depressive symptom rates


In [ ]:
"""
Youth Depression Risk Profiling Pipeline

This module combines preprocessing, model training/testing, hyperparameter
comparison, feature selection, and K-means clustering into reusable functions.

Designed for the Gachon University Data Science Mathematics team project.
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    silhouette_score,
)


def _make_onehot_encoder():
    """Create OneHotEncoder compatible with different scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def preprocess_youth_dataset(filepath, model_type="lr", encoding="cp949"):
    """
    Preprocess the 2024 Youth Life Survey dataset.

    Parameters
    ----------
    filepath : str
        Path to the CSV file.
    model_type : {"lr", "rf"}, default="lr"
        - "lr": applies log1p transformation and scaling for Logistic Regression.
        - "rf": applies log1p transformation but does not apply numerical scaling.
    encoding : str, default="cp949"
        CSV file encoding.

    Returns
    -------
    X_final : pandas.DataFrame
        Preprocessed feature matrix.
    y : pandas.Series
        Binary target variable.
    """
    df = pd.read_csv(filepath, encoding=encoding)
    df.columns = df.columns.str.strip()

    # Target conversion: 1 = no depressive symptoms -> 0, 2 = depressive symptoms -> 1
    df["Target"] = df["우울증상 유병 여부"].map({1: 0, 2: 1})
    y = df["Target"]

    # Remove target and burnout variable for policy interpretability
    drop_cols = ["우울증상 유병 여부", "Target", "최근 1년 소진(번아웃) 경험 여부"]
    X = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()

    # Derived feature: current working status
    # This prevents non-working status from being interpreted only as an ordinal scale value.
    if "경제활동상태" in X.columns:
        X["현재 근무 여부"] = X["경제활동상태"].apply(lambda x: 1 if x == 1 else 0)
    else:
        X["현재 근무 여부"] = 0

    # Ordinal remapping: larger value means higher frequency/intensity/status
    ordinal_mappings = {
        "현재 흡연 여부": {4: 1, 3: 2, 2: 3, 1: 4},
        "최근 1년간 음주 빈도": {7: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6},
        "외식 또는 매식 빈도": {5: 0, 4: 1, 3: 2, 2: 3, 1: 4},
        "일자리 불안정성 정도 - 현재의 일을 그만두거나 실직하더라도 나는 비슷한 돈을 벌 수 있는 직업을 쉽게 찾을 수 있을 것이다": {
            5: 1, 4: 2, 3: 3, 2: 4, 1: 5
        },
        "정치에 대한 관심 정도": {4: 1, 3: 2, 2: 3, 1: 4},
        "외출 빈도": {8: 1, 7: 2, 6: 3, 5: 4, 4: 5, 3: 6, 2: 7, 1: 8},
        "주관적 계층 인식": {5: 1, 4: 2, 3: 3, 2: 4, 1: 5},
    }

    standard_ordinal_cols = [
        "연령별", "가구원수", "최종학력", "음주 정도", "평소 규칙적 운동 여부",
        "고용 계약 기간", "총 고용 예상 기간", "지난 주 일자리 종사자 수", "재직 기간(범위)"
    ]

    # Employment-related ordinal variables with structural missing values
    if "정규근로시간 외 추가 근무" in X.columns:
        X["정규근로시간 외 추가 근무"] = X["정규근로시간 외 추가 근무"].map(
            {6: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6, 7: 1}
        ).fillna(0)

    if "정규 근로일 외 휴일 근무 횟수" in X.columns:
        X["정규 근로일 외 휴일 근무 횟수"] = X["정규 근로일 외 휴일 근무 횟수"].map(
            {0: 1, 1: 2, 2: 3}
        ).fillna(0)

    for col, mapping in ordinal_mappings.items():
        if col in X.columns:
            X[col] = X[col].fillna(0).map(mapping).fillna(0)

    for col in standard_ordinal_cols:
        if col in X.columns:
            X[col] = X[col].fillna(0)

    # Nominal variables: replace structural missing values with "Not Applicable"
    nominal_cols = [
        "성별", "지역별", "장애여부", "귀하의 현재 재학 상태를 응답해 주십시오.",
        "(복수 응답) 가구유형(1)",
        "국민기초생활보장제도(또는 맞춤형 급여) 수급 여부 및 경험",
        "돌봄 필요 가구원 유무", "혼인 상태", "부모 동거 여부",
        "공공 임대 주택 거주 경험", "현재 주거 점유 형태",
        "최근 1달 동안 같이 식사한 사람", "경제활동상태",
        "지난 주 수입을 목적으로 1시간 이상 일한 경험", "지난 주 종사상 지위",
        "고용 계약 기간 유무", "주휴 수당 수급 여부",
        "정규근로시간 외 추가 근무 시 추가 수당 수급 여부",
        "정규근로일 외 휴일 근무 시 추가 수당 수급 여부",
        "장시간 근로 경험(퇴근한 후부터 다음 날 출근하기까지의 시간이 11시간 미만)",
        "대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 - 가족^ 친척",
        "대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 - 이외",
        "대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무 – 이외",
        "현재 연애 여부(유배우 포함)", "향후 결혼 계획(유배우 포함)",
        "금융 채무 불이행자(신용불량자)에 해당 여부",
        "현재 근무 여부",
    ]

    valid_nominal_cols = [col for col in nominal_cols if col in X.columns]
    for col in valid_nominal_cols:
        X[col] = X[col].fillna("해당없음").astype(str)

    if valid_nominal_cols:
        ohe = _make_onehot_encoder()
        X_ohe = pd.DataFrame(
            ohe.fit_transform(X[valid_nominal_cols]),
            columns=ohe.get_feature_names_out(valid_nominal_cols),
            index=X.index,
        )
        ohe_cols = list(X_ohe.columns)
    else:
        X_ohe = pd.DataFrame(index=X.index)
        ohe_cols = []

    # Continuous variables: log1p transformation
    robust_cols = ["청년 연간소득 - 총 소득", "청년 기준 부채 총액", "청년 기준 재산총액"]
    standard_cols = ["월 평균 총생활비"]

    for col in robust_cols + standard_cols:
        if col in X.columns:
            X[col] = X[col].fillna(0)
            X[col] = np.log1p(X[col])

    X_non_nominal = X.drop(columns=valid_nominal_cols, errors="ignore")
    X_final = pd.concat([X_non_nominal, X_ohe], axis=1)

    # Make all columns numeric
    X_final = X_final.apply(pd.to_numeric, errors="coerce").fillna(0)

    if model_type == "lr":
        valid_robust_cols = [col for col in robust_cols if col in X_final.columns]
        if valid_robust_cols:
            X_final[valid_robust_cols] = RobustScaler().fit_transform(X_final[valid_robust_cols])

        valid_standard_cols = [
            col for col in standard_cols + standard_ordinal_cols + list(ordinal_mappings.keys())
            if col in X_final.columns and col not in valid_robust_cols and col not in ohe_cols
        ]
        if valid_standard_cols:
            X_final[valid_standard_cols] = StandardScaler().fit_transform(X_final[valid_standard_cols])

    elif model_type == "rf":
        # Tree-based model: scaling is not required
        pass
    else:
        raise ValueError("model_type must be either 'lr' or 'rf'.")

    return X_final, y


def evaluate_predictions(y_true, y_prob, threshold=0.5):
    """
    Evaluate binary classification predictions.

    Parameters
    ----------
    y_true : array-like
        True labels.
    y_prob : array-like
        Predicted probability of the positive class.
    threshold : float, default=0.5
        Decision threshold.

    Returns
    -------
    dict
        Evaluation metrics.
    """
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "threshold": threshold,
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


def cross_validated_probabilities(model, X, y, cv):
    """
    Return out-of-fold predicted probabilities for a given model.
    """
    probs = np.zeros(len(y))
    for train_idx, val_idx in cv.split(X, y):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        probs[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    return probs


def run_model_experiments(
    filepath,
    encoding="cp949",
    scoring="f1_score",
    random_state=42,
    lr_c_values=None,
    rf_param_grid=None,
    thresholds=None,
):
    """
    Run preprocessing, model training/testing, hyperparameter comparison,
    and evaluation under a single top-level function.

    This function satisfies the open-source SW contribution requirement:
    for a cleaned dataset, it compares preprocessing, model algorithms,
    model parameters, and evaluation metrics, then returns the top five
    and best combination.

    Parameters
    ----------
    filepath : str
        Path to the CSV dataset.
    encoding : str, default="cp949"
        CSV encoding.
    scoring : {"accuracy", "precision", "recall", "f1_score", "auc"}, default="recall"
        Metric used to rank model combinations.
    random_state : int, default=42
        Random seed.
    lr_c_values : list, optional
        Candidate C values for Logistic Regression.
    rf_param_grid : dict, optional
        Candidate hyperparameters for Random Forest.
    thresholds : list, optional
        Candidate decision thresholds.

    Returns
    -------
    results_df : pandas.DataFrame
        Model comparison table sorted by the scoring metric.
    best_info : dict
        Information about the best model combination.
    processed_data : dict
        Preprocessed datasets for LR and RF.
    """
    if lr_c_values is None:
        lr_c_values = [0.001, 0.01, 0.1, 1, 10]

    if rf_param_grid is None:
        rf_param_grid = {
            "n_estimators": [100, 200],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5],
        }

    if thresholds is None:
        thresholds = [0.21, 0.23, 0.30, 0.50, 0.62, 0.69]

    X_lr, y_lr = preprocess_youth_dataset(filepath, model_type="lr", encoding=encoding)
    X_rf, y_rf = preprocess_youth_dataset(filepath, model_type="rf", encoding=encoding)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

    results = []
    fitted_models = {}

    # Logistic Regression experiments
    for C in lr_c_values:
        model = LogisticRegression(
            C=C,
            max_iter=3000,
            class_weight="balanced",
            random_state=random_state,
        )
        probs = cross_validated_probabilities(model, X_lr, y_lr, cv)

        for th in thresholds:
            metrics = evaluate_predictions(y_lr, probs, threshold=th)
            row = {
                "preprocessing": "log1p + RobustScaler/StandardScaler + OHE",
                "model": "LogisticRegression",
                "params": {"C": C, "class_weight": "balanced"},
                "threshold": th,
                "accuracy": metrics["accuracy"],
                "precision": metrics["precision"],
                "recall": metrics["recall"],
                "f1_score": metrics["f1_score"],
                "auc": metrics["auc"],
            }
            results.append(row)

        model.fit(X_lr, y_lr)
        fitted_models[("LogisticRegression", str({"C": C}), "lr")] = model

    # Random Forest experiments
    for n_estimators in rf_param_grid["n_estimators"]:
        for max_depth in rf_param_grid["max_depth"]:
            for min_samples_split in rf_param_grid["min_samples_split"]:
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    class_weight="balanced",
                    random_state=random_state,
                    n_jobs=-1,
                )
                probs = cross_validated_probabilities(model, X_rf, y_rf, cv)

                for th in thresholds:
                    metrics = evaluate_predictions(y_rf, probs, threshold=th)
                    row = {
                        "preprocessing": "log1p + OHE (no scaling)",
                        "model": "RandomForest",
                        "params": {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_split": min_samples_split,
                            "class_weight": "balanced",
                        },
                        "threshold": th,
                        "accuracy": metrics["accuracy"],
                        "precision": metrics["precision"],
                        "recall": metrics["recall"],
                        "f1_score": metrics["f1_score"],
                        "auc": metrics["auc"],
                    }
                    results.append(row)

                model.fit(X_rf, y_rf)
                fitted_models[("RandomForest", str({
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                }), "rf")] = model

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by=scoring, ascending=False).reset_index(drop=True)

    best_row = results_df.iloc[0].to_dict()
    best_model_name = best_row["model"]
    best_params = best_row["params"]

    # Fit and store best model
    if best_model_name == "LogisticRegression":
        best_X, best_y = X_lr, y_lr
        best_model = LogisticRegression(
            C=best_params["C"],
            max_iter=3000,
            class_weight="balanced",
            random_state=random_state,
        )
    else:
        best_X, best_y = X_rf, y_rf
        best_model = RandomForestClassifier(
            n_estimators=best_params["n_estimators"],
            max_depth=best_params["max_depth"],
            min_samples_split=best_params["min_samples_split"],
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )

    best_model.fit(best_X, best_y)

    best_info = {
        "best_row": best_row,
        "best_model": best_model,
        "top5": results_df.head(5),
    }

    processed_data = {
        "X_lr": X_lr,
        "y_lr": y_lr,
        "X_rf": X_rf,
        "y_rf": y_rf,
    }

    return results_df, best_info, processed_data


def get_feature_importance(model, X, top_n=20):
    """
    Extract feature importance from Logistic Regression coefficients
    or Random Forest feature importances.
    """
    if hasattr(model, "coef_"):
        importance = np.abs(model.coef_[0])
    elif hasattr(model, "feature_importances_"):
        importance = model.feature_importances_
    else:
        raise ValueError("The model does not provide feature importance.")

    importance_df = pd.DataFrame({
        "feature": X.columns,
        "importance": importance,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return importance_df.head(top_n)


def run_kmeans_clustering(
    X,
    y,
    feature_cols,
    k_range=range(2, 11),
    random_state=42,
):
    """
    Perform K-means clustering using selected features and analyze
    depressive symptom rate by cluster.

    Parameters
    ----------
    X : pandas.DataFrame
        Preprocessed feature matrix.
    y : pandas.Series
        Binary target variable.
    feature_cols : list
        Base feature names or keywords to select clustering features.
    k_range : iterable, default=range(2, 11)
        Candidate K values.
    random_state : int, default=42
        Random seed.

    Returns
    -------
    clustering_result : dict
        K search result, final labels, and cluster depression rates.
    """
    selected_cols = []
    for base_feat in feature_cols:
        # Flexible matching for OHE-generated columns and Korean punctuation variants
        if "이외" in base_feat:
            matched = [
                col for col in X.columns
                if "교류하는 사람의 유무" in col and "이외" in col
            ]
        else:
            matched = [col for col in X.columns if base_feat in col]

        selected_cols.extend(matched)

    selected_cols = sorted(list(set(selected_cols)))

    if not selected_cols:
        raise ValueError("No matching feature columns were found for clustering.")

    X_cluster = X[selected_cols]

    k_results = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(X_cluster)
        k_results.append({
            "k": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(X_cluster, labels),
        })

    k_results_df = pd.DataFrame(k_results)
    best_k = int(k_results_df.loc[k_results_df["silhouette"].idxmax(), "k"])

    final_kmeans = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
    labels = final_kmeans.fit_predict(X_cluster)

    cluster_df = pd.DataFrame({
        "cluster": labels,
        "target": y.values,
    })

    cluster_summary = cluster_df.groupby("cluster")["target"].agg(
        count="count",
        depressive_symptom_rate="mean"
    ).reset_index()

    return {
        "selected_columns": selected_cols,
        "k_results": k_results_df,
        "best_k": best_k,
        "labels": labels,
        "cluster_summary": cluster_summary,
        "model": final_kmeans,
    }


def run_depression_risk_pipeline(
    filepath,
    encoding="cp949",
    scoring="recall",
    clustering_features=None,
):
    """
    Complete project pipeline:
    1. Preprocess data
    2. Compare model/preprocessing/parameter combinations
    3. Select the best model
    4. Extract feature importance
    5. Run K-means clustering with selected project features

    Parameters
    ----------
    filepath : str
        Path to the CSV file.
    encoding : str, default="cp949"
        CSV encoding.
    scoring : str, default="recall"
        Metric used to rank models.
    clustering_features : list, optional
        Feature names used for clustering.

    Returns
    -------
    output : dict
        Results from model experiments, feature importance, and clustering.
    """
    if clustering_features is None:
        clustering_features = [
            "평소 본인에 대한 건강 인식",
            "외출 빈도",
            "청년 기준 부채 총액",
            "정치에 대한 관심 정도",
            "가구원수",
            "주관적 계층 인식",
            "외식 또는 매식 빈도",
            "성별",
            "대면^ 인터넷(SNS)^ 전화 등의 방식으로 교류하는 사람의 유무",
            "최근 1달 동안 같이 식사한 사람",
        ]

    results_df, best_info, processed_data = run_model_experiments(
        filepath=filepath,
        encoding=encoding,
        scoring=scoring,
    )

    best_model = best_info["best_model"]
    best_model_name = best_info["best_row"]["model"]

    if best_model_name == "LogisticRegression":
        X_for_importance = processed_data["X_lr"]
        y_for_cluster = processed_data["y_lr"]
        X_for_cluster = processed_data["X_lr"]
    else:
        X_for_importance = processed_data["X_rf"]
        y_for_cluster = processed_data["y_lr"]
        # K-means is distance-based, so use the scaled LR version for clustering
        X_for_cluster = processed_data["X_lr"]

    importance_df = get_feature_importance(best_model, X_for_importance, top_n=20)

    clustering_result = run_kmeans_clustering(
        X=X_for_cluster,
        y=y_for_cluster,
        feature_cols=clustering_features,
    )

    output = {
        "model_results": results_df,
        "top5_combinations": best_info["top5"],
        "best_combination": best_info["best_row"],
        "best_model": best_model,
        "feature_importance": importance_df,
        "clustering_result": clustering_result,
    }

    return output


if __name__ == "__main__":
    # Example usage:
    # filepath = "2024_청년삶실태조사.csv"
    # output = run_depression_risk_pipeline(filepath)
    # print(output["top5_combinations"])
    # print(output["best_combination"])
    # print(output["clustering_result"]["cluster_summary"])
    pass

## Example Usage

```python
filepath = "2024_청년삶실태조사.csv"

output = run_depression_risk_pipeline(
    filepath=filepath,
    encoding="cp949",
    scoring="f1_score"
)

output["top5_combinations"]
```


In [2]:
# Example run
# filepath = "2024_청년삶실태조사.csv"
# output = run_depression_risk_pipeline(filepath=filepath, scoring="recall")
# display(output["top5_combinations"])
# display(output["feature_importance"])
# display(output["clustering_result"]["k_results"])
# display(output["clustering_result"]["cluster_summary"])